In [4]:
import json
import pandas as pd
from pathlib import Path

# paths
test_path = "./stage2_cv_results/fixed_splits/test_500.json"
winning_path = "./stage2_cv_results_final/winning_test_result.json"
joint_path = "./stage2_cv_results_final/joint_committee_biomed_r1_results.json"

# label mapping
id_to_label = {
    0: "no",
    1: "maybe",
    2: "yes",
}

 # load files
with open(test_path, "r") as f:
    test_500 = json.load(f)

with open(winning_path, "r") as f:
    winning_result = json.load(f)

with open(joint_path, "r") as f:
    joint_result = json.load(f)

# winning pipeline predictions
winning_preds = [id_to_label[p] for p in winning_result["test_preds"]]

winning_rows = []
for i, row in enumerate(test_500):
    pubmed_id = row["ID"]
    y_true = row["final_decision"]
    y_pred = winning_preds[i]
    correct = int(y_true == y_pred)

    winning_rows.append({
        "pubmed_id": pubmed_id,
        "y_true": y_true,
        "y_pred": y_pred,
        "correct": correct,
    })

winning_df = pd.DataFrame(winning_rows)

# joint committee predictions
joint_preds = winning_preds.copy()

for item in joint_result["reasoning_rows"]:
    row_id = item["row_id"]
    final_label = item["final_label"]

    joint_preds[row_id] = final_label

joint_rows = []
for i, row in enumerate(test_500):
    pubmed_id = row["ID"]
    y_true = row["final_decision"]
    y_pred = joint_preds[i]
    correct = int(y_true == y_pred)

    joint_rows.append({
        "pubmed_id": pubmed_id,
        "y_true": y_true,
        "y_pred": y_pred,
        "correct": correct,
    })

joint_df = pd.DataFrame(joint_rows)

# save csvs
output_dir = Path("./stage2_cv_results_final")
output_dir.mkdir(parents=True, exist_ok=True)

winning_csv_path = output_dir / "BiomedClinicalBERT_winning_pipeline_results.csv"
joint_csv_path = output_dir / "BiomedxR1_joint_committee_results.csv"

winning_df.to_csv(winning_csv_path, index=False)
joint_df.to_csv(joint_csv_path, index=False)

print(f"Saved winning pipeline CSV to: {winning_csv_path}")
print(f"Saved joint committee CSV to: {joint_csv_path}")

print("\nWinning pipeline preview:")
print(winning_df.head())

print("\nJoint committee preview:")
print(joint_df.head())

Saved winning pipeline CSV to: stage2_cv_results_final/BiomedClinicalBERT_winning_pipeline_results.csv
Saved joint committee CSV to: stage2_cv_results_final/BiomedxR1_joint_committee_results.csv

Winning pipeline preview:
  pubmed_id y_true y_pred  correct
0  21645374    yes    yes        1
1   9488747    yes    yes        1
2  17208539     no    yes        0
3  23831910    yes    yes        1
4  26037986  maybe    yes        0

Joint committee preview:
  pubmed_id y_true y_pred  correct
0  21645374    yes    yes        1
1   9488747    yes    yes        1
2  17208539     no    yes        0
3  23831910    yes    yes        1
4  26037986  maybe    yes        0
